# Using the Allmaps API with the Digital Commonwealth JSON API

This notebook shows how to retrieve object metadata for a map in Digital Commonwealth / LMEC by climbing up from the Allmaps API to the Digital Commonwealth JSON API.

Let's say we have this set of Allmaps XYZ tiles in a web map and we want to programmatically add metadata to it:

    https://allmaps.xyz/maps/10c1664f2c9d2cff/{z}/{x}/{y}.png

To determine where these tiles came from, we can combine the Allmaps API with the Digital Commonwealth (DC) JSON API and climb from the Allmaps image ID `10c1664f2c9d2cff` all the way up to the DC manifest for the parent object. Metadata for these XYZ tiles, including author, title, date, and more, are all available with the parent object.

This process can be broken up into four steps:

1. get the Allmaps ID from the XYZ tiles
2. pass the Allmaps ID to the Allmaps API's base URL to return the DC image ID
3. pass the DC image ID to the DC JSON API's base URL to return its parent object's manifest ID
4. pass manifest ID to LMEC collections base URL to return object metadata

# 1. get the Allmaps ID from the XYZ tiles

In [ ]:
import requests
import json

allmapsId = requests.get("https://api.allmaps.org/maps/10c1664f2c9d2cff/manifests").json()[0]['imageId']

print(allmapsId)

# 2. pass the Allmaps ID to the Allmaps API's base URL to return the DC image ID

In [ ]:
baseAnnotationURL = "https://annotations.allmaps.org/images/"
fullAnnotationURL = baseAnnotationURL+allmapsId
annotationRequest = requests.get(fullAnnotationURL)
imageDC = annotationRequest.json()['items'][0]['target']['service'][0]['@id']
imageIdDC = (imageDC[-9:])

print(imageIdDC)

# 3. pass the DC image ID to the DC JSON API's base URL to return its parent object's manifest ID

In [ ]:
imageParent = requests.get("https://www.digitalcommonwealth.org/search/commonwealth:" + imageIdDC + ".json")
manifestId = imageParent.json()['data']['attributes']['is_file_set_of_ssim'][0] + "/manifest.json"

print(manifestId)

# 4. pass manifest ID to LMEC collections base URL to return object metadata

In [ ]:
collectionsRecordManifest = "https://collections.leventhalmap.org/search/" + manifestId
metadataRequest = requests.get(collectionsRecordManifest)
metadata = json.dumps(metadataRequest.json(), indent=2)

print(metadata)